# Duka Electronics — Sales Exploration (January–June 2026)

## Executive Summary

- **[N]** orders generated **KES [X]** in total revenue during January–June 2026.
- **[Branch]** was the leading branch, contributing **KES [X]** across **[N]** orders.
- **[Product]** ranked first by revenue at **KES [X]**, while **[Product]** led units sold with **[N]** units.
- Weekend orders generated **[X]%** of revenue, compared with **[Y]%** from weekday orders.
- **[Month]** was the strongest month at **KES [X]**, exceeding the weakest month, **[Month]**, by **[X]%**.

> Replace every bracketed value after running the analysis on the original generated dataset. Do not submit placeholders.


## 1. Load data and first-five-minutes inspection
This section documents shape, data types, missing values, date range and key categories before analysis.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

sales = pd.read_csv('duka_sales.csv', parse_dates=['order_date'])
print('Shape:', sales.shape)
print('\nData types:\n', sales.dtypes)
print('\nMissing values:\n', sales.isna().sum())
print('\nDate range:', sales['order_date'].min(), 'to', sales['order_date'].max())
for col in ['branch','product','channel','region']:
    if col in sales.columns:
        print(f'\\n{col} categories:', sales[col].value_counts(dropna=False).to_dict())
display(sales.head())
display(sales.describe(include='all').T)


## 2. Derived columns
Revenue is calculated as quantity multiplied by unit price. Price band uses the assignment threshold of KES 28,000.

In [ ]:
required = {'order_date','branch','product','channel','region','quantity','unit_price'}
missing = required - set(sales.columns)
if missing:
    raise ValueError(f'Duka file is missing expected columns: {sorted(missing)}. Check the original practical generator/schema.')
sales['revenue'] = sales['quantity'] * sales['unit_price']
sales['month'] = sales['order_date'].dt.month_name()
sales['weekday'] = sales['order_date'].dt.day_name()
sales['is_weekend'] = sales['order_date'].dt.dayofweek >= 5
sales['price_band'] = np.where(sales['unit_price'] >= 28000, 'Premium', 'Value')
display(sales.head())


## Task 3 — Branch ranking by revenue and orders
Compare the two rankings and explain why high-value baskets can change revenue rank without changing order volume rank.

In [ ]:
branch_rank = sales.groupby('branch').agg(total_revenue=('revenue','sum'), orders=('revenue','size')).sort_values('total_revenue', ascending=False)
branch_rank['revenue_rank'] = branch_rank['total_revenue'].rank(method='min', ascending=False).astype(int)
branch_rank['order_rank'] = branch_rank['orders'].rank(method='min', ascending=False).astype(int)
display(branch_rank)
print('Are rankings identical?', branch_rank['revenue_rank'].equals(branch_rank['order_rank']))


## Task 4 — Top products by revenue and units sold
The lists may differ because a product can sell fewer units at a higher price, while another sells many units at a lower price.

In [ ]:
product_rank = sales.groupby('product').agg(total_revenue=('revenue','sum'), units_sold=('quantity','sum'))
print('Top 3 by revenue')
display(product_rank.nlargest(3, 'total_revenue'))
print('Top 3 by units sold')
display(product_rank.nlargest(3, 'units_sold'))


## Task 5 — Weekend versus weekday orders

In [ ]:
sales['day_group'] = np.where(sales['is_weekend'], 'Weekend', 'Weekday')
weekend_comparison = sales.groupby('day_group').agg(number_of_orders=('revenue','size'), total_revenue=('revenue','sum'), average_order_revenue=('revenue','mean'))
display(weekend_comparison)


## Task 6 — Best and worst revenue months
Percentage difference is calculated relative to the worst month.

In [ ]:
monthly = sales.groupby('month', sort=False)['revenue'].sum().sort_values(ascending=False)
best_month, best_revenue = monthly.index[0], monthly.iloc[0]
worst_month, worst_revenue = monthly.index[-1], monthly.iloc[-1]
pct_difference = ((best_revenue - worst_revenue) / worst_revenue) * 100
print(f'Best month: {best_month} — KES {best_revenue:,.2f}')
print(f'Worst month: {worst_month} — KES {worst_revenue:,.2f}')
print(f'Best month exceeded worst by {pct_difference:.2f}%')
display(monthly)


## Task 7 — Premium online orders in Nairobi and Mombasa
Filter Premium orders sold through the Online channel in the two specified cities and export without an index.

In [ ]:
premium_online = sales[(sales['price_band'] == 'Premium') & (sales['channel'].astype(str).str.lower() == 'online') & (sales['region'].isin(['Nairobi','Mombasa']))].copy()
print('Matching orders:', len(premium_online))
print('Total revenue: KES', f'{premium_online["revenue"].sum():,.2f}')
Path('outputs').mkdir(exist_ok=True)
premium_online.to_csv('outputs/premium_online_coast_capital.csv', index=False)


## Task 8 — Excel branch summary
The workbook is generated with a SUMIFS branch revenue table, XLOOKUP region column and a branch-by-channel revenue summary. Recalculate in Excel to refresh formulas.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

Path('excel').mkdir(exist_ok=True)
wb = Workbook()
ws = wb.active
ws.title = 'Sales Data'
ws.append(list(sales.columns))
for row in sales.itertuples(index=False, name=None):
    ws.append([v.to_pydatetime() if hasattr(v, 'to_pydatetime') else v for v in row])
ws.freeze_panes = 'A2'
headers = {c.value:i+1 for i,c in enumerate(ws[1])}
branch_col = get_column_letter(headers['branch'])
revenue_col = get_column_letter(headers['revenue'])
region_col = get_column_letter(headers['region'])
channel_col = get_column_letter(headers['channel'])
summary = wb.create_sheet('Branch Summary')
summary.append(['Branch','Revenue (SUMIFS)','Region (XLOOKUP)','Orders'])
branches = sorted(sales['branch'].dropna().unique())
for r, branch in enumerate(branches, start=2):
    summary.cell(r,1,branch)
    summary.cell(r,2,f'=SUMIFS(\'Sales Data\'!${revenue_col}:${revenue_col},\'Sales Data\'!${branch_col}:${branch_col},A{r})')
    summary.cell(r,3,f'=XLOOKUP(A{r},\'Sales Data\'!${branch_col}:${branch_col},\'Sales Data\'!${region_col}:${region_col},"Not found")')
    summary.cell(r,4,f'=COUNTIF(\'Sales Data\'!${branch_col}:${branch_col},A{r})')
pivot = wb.create_sheet('Branch Channel Pivot')
channels = sorted(sales['channel'].dropna().unique())
pivot.append(['Branch'] + channels + ['Grand Total'])
for r, branch in enumerate(branches, start=2):
    pivot.cell(r,1,branch)
    for c, channel in enumerate(channels, start=2):
        col = get_column_letter(c)
        pivot.cell(r,c,f'=SUMIFS(\'Sales Data\'!${revenue_col}:${revenue_col},\'Sales Data\'!${branch_col}:${branch_col},$A{r},\'Sales Data\'!${channel_col}:${channel_col},{col}$1)')
    last = get_column_letter(len(channels)+1)
    pivot.cell(r,len(channels)+2,f'=SUM(B{r}:{last}{r})')
for sh in [ws, summary, pivot]:
    for cell in sh[1]:
        cell.font = Font(bold=True, color='FFFFFF')
        cell.fill = PatternFill('solid', fgColor='16324F')
    for col in sh.columns:
        sh.column_dimensions[get_column_letter(col[0].column)].width = min(max(max(len(str(x.value or '')) for x in col)+2, 14), 32)
wb.save('excel/duka_branch_summary.xlsx')

pandas_total = sales['revenue'].sum()
print('Pandas total revenue: KES', f'{pandas_total:,.2f}')
print('Excel formulas are included; open in Excel to calculate and compare the Grand Total with this pandas total.')


## Task 9 — Executive Summary validation
Use the computed tables above to replace all five numeric placeholders in the Executive Summary at the top before submission.